# Web Skin 최종 후보 — PMG·B0·256 Test 1회 평가와 패키징

Validation 비교와 실행 비용을 함께 고려해 고정한 PMG·EfficientNet-B0·256 후보를 다시
학습하지 않고 Test에서 최종 평가합니다. 선택 모델의 SHA-256을 먼저 확인하며 Test 결과로
설정을 변경하지 않습니다. 평가 후 모델, 클래스 순서, 전처리 계약과 결과를 ZIP으로 묶습니다.


## 1. 고정된 후보

- 방법: PMG(Progressive Multi-Granularity) adaptation
- Backbone: ImageNet EfficientNet-B0
- 입력: 256×256 RGB float32, 픽셀 0–255
- Loss / Optimizer: Cross Entropy / Adam
- 학습: Stage 1 15 epoch + Stage 2 10 epoch — 이미 완료
- 선택 체크포인트: `pmg_b0_256_ce_seed_42/.../stage2_best.keras`
- 고정 Validation: Accuracy 0.85, Macro F1 0.8473279632397033
- Test 평가: 이 노트북에서 처음 1회

PMG 모델은 네 branch의 logit을 출력합니다. 후보 ZIP의 `inference.py`가 네 출력을 합산한 뒤
softmax를 적용합니다. 중단 후에는 처음 출력된 결과 폴더를 `RESUME_DIR`에 넣어야 저장된
Test 결과를 재사용합니다.


In [ ]:
DOMAIN = 'web_skin'
PROJECT_ROOT = '/content/drive/MyDrive/mediflow_Project'
DATA_ZIP = '/content/drive/MyDrive/mediflow_Project/datasets/web_skin_datasets.zip'
AUDIT_DIR = ''
EXPECTED_DATA_SHA256 = 'f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d'
RESUME_DIR = ''
MODE = 'web_skin_pmg_final'
SEED = 42
SEEDS = [42]
BATCH_SIZE = 32
STAGE1_EPOCHS = 15  # 학습하지 않음
STAGE2_EPOCHS = 10  # 학습하지 않음
EXTENSION_EPOCHS = 1  # 사용하지 않음
TRAIN_VARIANT = 'augmented'
PARENT_RUN_DIR = (
    '/content/drive/MyDrive/mediflow_Project/2_results/web_skin/'
    'web_skin_paper_suite_20260922_124758_717b465e'
)
PARENT_MODEL_SHA256 = '83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4'


## 2. Drive와 실행 환경

`PARENT_RUN_DIR`의 PMG B0·256 모델과 데이터 ZIP이 남아 있어야 합니다. 이 노트북은
학습하지 않으며 Test 추론과 패키징만 수행합니다. GPU를 선택하면 평가가 더 빠릅니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install tensorflow==2.20.0 keras==3.13.2 pandas matplotlib pillow tqdm
import tensorflow as tf
import keras
if tf.__version__ != '2.20.0' or keras.__version__ != '3.13.2':
    raise RuntimeError('Colab 세션을 다시 시작한 뒤 처음부터 실행하세요.')


## 3. 재현 코드

수정하지 않습니다. 실행 코드는 결과와 최종 후보 패키지에 함께 저장됩니다.


In [ ]:
import sys, types
SOURCES = {'common_engine': '"""Sequential domain-configured experiments; validation selection precedes any test inference.\n\nThe Colab notebook embeds an exact copy of this module so no repository checkout\nis needed in Colab. Completed trials are reused only after artifact verification.\nInterrupted attempts are preserved and restarted, not resumed mid-epoch.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\n\nPROTOCOL = "mediflow_common_v1"\nTRIALS = [\n    {"id": "b0_224_ce", "backbone": "B0", "size": 224, "loss": "ce"},\n    {"id": "b0_256_ce", "backbone": "B0", "size": 256, "loss": "ce"},\n    {"id": "b0_256_ls005", "backbone": "B0", "size": 256, "loss": "ls005"},\n    {"id": "b0_256_focal15", "backbone": "B0", "size": 256, "loss": "focal15"},\n    {"id": "b1_256_ls005", "backbone": "B1", "size": 256, "loss": "ls005"},\n]\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(".json-" + uuid.uuid4().hex[:12] + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef classification_metrics(truth, probabilities, count=5):\n    truth = np.asarray(truth, dtype=np.int64)\n    probabilities = np.asarray(probabilities)\n    if (\n        truth.ndim != 1\n        or not len(truth)\n        or probabilities.shape != (len(truth), count)\n        or not np.isfinite(probabilities).all()\n        or np.any(truth < 0)\n        or np.any(truth >= count)\n    ):\n        raise ValueError("Invalid evaluation arrays")\n    predictions = probabilities.argmax(axis=1)\n    cm = np.bincount(count * truth + predictions, minlength=count * count).reshape(count, count)\n    tp = np.diag(cm).astype(float)\n    precision = np.divide(tp, cm.sum(0), out=np.zeros(count), where=cm.sum(0) != 0)\n    recall = np.divide(tp, cm.sum(1), out=np.zeros(count), where=cm.sum(1) != 0)\n    f1 = np.divide(\n        2 * precision * recall,\n        precision + recall,\n        out=np.zeros(count),\n        where=precision + recall != 0,\n    )\n    return {\n        "accuracy": float(np.mean(truth == predictions)),\n        "macro_f1": float(f1.mean()),\n        "class_f1": f1.tolist(),\n        "precision": precision.tolist(),\n        "recall": recall.tolist(),\n        "support": cm.sum(1).tolist(),\n        "confusion_matrix": cm.tolist(),\n        "count": len(truth),\n    }\n\n\ndef predict_dataset(model, dataset):\n    truth, probabilities = [], []\n    for images, labels in dataset:\n        probabilities.extend(model(images, training=False).numpy())\n        truth.extend(np.argmax(labels.numpy(), axis=1))\n    return np.asarray(truth, dtype=np.int64), np.asarray(probabilities)\n\n\ndef save_predictions(path, paths, truth, probabilities):\n    if len(paths) != len(truth):\n        raise ValueError("File order and prediction count differ")\n    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "path",\n                "true_index",\n                "pred_index",\n                *[f"prob_C{i}" for i in range(probabilities.shape[1])],\n            ]\n        )\n        for name, target, probs in zip(paths, truth, probabilities, strict=True):\n            writer.writerow([name, int(target), int(probs.argmax()), *map(float, probs)])\n\n\ndef loss_function(name):\n    if name == "ce":\n        return keras.losses.CategoricalCrossentropy()\n    if name == "ls005":\n        return keras.losses.CategoricalCrossentropy(label_smoothing=0.05)\n    if name == "focal15":\n        return keras.losses.CategoricalFocalCrossentropy(alpha=1.0, gamma=1.5)\n    raise ValueError(name)\n\n\ndef build_model(spec):\n    builder = {\n        "B0": keras.applications.EfficientNetB0,\n        "B1": keras.applications.EfficientNetB1,\n        "V2S": keras.applications.EfficientNetV2S,\n    }\n    size = spec["size"]\n    backbone = builder[spec["backbone"]](\n        include_top=False, weights="imagenet", input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = backbone(inputs, training=False)\n    features = keras.layers.GlobalAveragePooling2D()(features)\n    features = keras.layers.Dropout(0.3)(features)\n    outputs = keras.layers.Dense(spec["class_count"], activation="softmax")(features)\n    model = keras.Model(inputs, outputs)\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-4),\n        loss=loss_function(spec["loss"]),\n        metrics=["accuracy"],\n    )\n    return model\n\n\ndef configure_partial(model, loss_name):\n    backbones = [\n        layer\n        for layer in model.layers\n        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()\n    ]\n    if len(backbones) != 1:\n        raise ValueError("Expected one EfficientNet backbone")\n    backbone = backbones[0]\n    backbone.trainable = True\n    for index, layer in enumerate(backbone.layers):\n        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(\n            layer, keras.layers.BatchNormalization\n        )\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-5), loss=loss_function(loss_name), metrics=["accuracy"]\n    )\n    return [layer.name for layer in backbone.layers if layer.trainable]\n\n\nclass HistoryBackup(keras.callbacks.Callback):\n    def __init__(self, path):\n        super().__init__()\n        self.path = path\n        self.values = {}\n\n    def on_epoch_end(self, epoch, logs=None):\n        for key, value in (logs or {}).items():\n            self.values.setdefault(key, []).append(float(value))\n        write_json(self.path, self.values)\n\n\ndef fit_stage(model, train, val, directory, name, epochs):\n    best = directory / (name + "_best.keras")\n    callbacks = [\n        keras.callbacks.ModelCheckpoint(\n            str(best), monitor="val_accuracy", mode="max", save_best_only=True\n        ),\n        keras.callbacks.CSVLogger(str(directory / (name + "_log.csv"))),\n        HistoryBackup(directory / (name + "_history.json")),\n        keras.callbacks.TerminateOnNaN(),\n    ]\n    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks, verbose=2)\n    values = {key: [float(v) for v in seq] for key, seq in history.history.items()}\n    if len(values.get("val_accuracy", [])) != epochs or not all(\n        np.isfinite(seq).all() for seq in values.values()\n    ):\n        raise RuntimeError("Incomplete or non-finite training; attempt retained")\n    model.save(directory / (name + "_last.keras"))\n    return values, best\n\n\ndef checkpoint_choice(baseline_score, new_score):\n    """Keep the earlier/simpler checkpoint on ties."""\n    return new_score > baseline_score\n\n\ndef cached_record(root, trial_id, signature):\n    trial_dir = Path(root) / trial_id\n    marker = trial_dir / "completed.json"\n    if not marker.exists():\n        return None\n    record = read_json(marker)\n    if record["signature"] != signature:\n        raise ValueError("Resume settings differ; use a new suite directory")\n    for relative, digest in record["artifact_hashes"].items():\n        target = (trial_dir / relative).resolve()\n        if not target.is_relative_to(trial_dir.resolve()) or file_hash(target) != digest:\n            raise ValueError("Completed artifact changed or corrupted: " + relative)\n    return record\n\n\ndef evaluate_to_files(model, dataset, paths, directory, prefix):\n    truth, probabilities = predict_dataset(model, dataset)\n    metrics = classification_metrics(truth, probabilities, probabilities.shape[1])\n    write_json(directory / (prefix + "_metrics.json"), metrics)\n    save_predictions(directory / (prefix + "_predictions.csv"), paths, truth, probabilities)\n    return metrics\n\n\ndef run_trial(spec, dataset_factory, root, signature, seed=42, epochs1=15, epochs2=10):\n    cached = cached_record(root, spec["id"], signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / spec["id"] / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    write_json(directory / "spec.json", spec)\n    train, _ = dataset_factory("train", spec["size"], True)\n    val, paths = dataset_factory("val", spec["size"], False)\n    started = time.monotonic()\n    model = build_model(spec)\n    h1, best1 = fit_stage(model, train, val, directory, "stage1", epochs1)\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(best1, compile=False)\n    trainable = []\n    h2 = {key: [] for key in h1}\n    best2 = best1\n    if epochs2:\n        trainable = configure_partial(model, spec["loss"])\n        h2, best2 = fit_stage(model, train, val, directory, "stage2", epochs2)\n    del model\n    selected_stage = (\n        "stage2"\n        if checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"], default=-1.0))\n        else "stage1"\n    )\n    selected = best2 if selected_stage == "stage2" else best1\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": spec["id"],\n        "spec": spec,\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": selected_stage,\n        "validation": metrics,\n        "stage1_best_val": max(h1["val_accuracy"]),\n        "stage2_best_val": max(h2["val_accuracy"]) if h2["val_accuracy"] else None,\n        "training_seconds": time.monotonic() - started,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "trainable_backbone_layers": trainable,\n        "history": {key: h1[key] + h2[key] for key in h1},\n        "stage_boundary": len(h1["accuracy"]),\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef finish_record(directory, record):\n    write_json(directory / "record.json", record)\n    record["artifact_hashes"] = {\n        str(path.relative_to(directory.parent)): file_hash(path)\n        for path in directory.iterdir()\n        if path.is_file()\n    }\n    write_json(directory.parent / "completed.json", record)\n\n\ndef selected_model_path(root, record):\n    return Path(root) / record["id"] / record["attempt"] / record["selected_model"]\n\n\ndef extend_b1(parent, dataset_factory, root, signature, seed=42, epochs=5):\n    trial_id = "b1_256_ls005_extend5"\n    cached = cached_record(root, trial_id, signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / trial_id / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    parent_dir = Path(root) / parent["id"] / parent["attempt"]\n    # Continue from epoch 10\'s LAST checkpoint, including optimizer state.\n    source = parent_dir / "stage2_last.keras"\n    model = keras.models.load_model(source)\n    if model.optimizer is None:\n        raise ValueError("Extension requires saved optimizer")\n    train, _ = dataset_factory("train", 256, True)\n    val, paths = dataset_factory("val", 256, False)\n    started = time.monotonic()\n    history, best = fit_stage(model, train, val, directory, "extension", epochs)\n    del model\n    parent_best = selected_model_path(root, parent)\n    parent_score = max(parent["stage1_best_val"], parent["stage2_best_val"])\n    keep_extension = checkpoint_choice(parent_score, max(history["val_accuracy"]))\n    selected = directory / "selected.keras"\n    import shutil\n\n    shutil.copyfile(best if keep_extension else parent_best, selected)\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": trial_id,\n        "spec": {**parent["spec"], "id": trial_id},\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": "extension" if keep_extension else "parent_" + parent["selected_stage"],\n        "validation": metrics,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "training_seconds": time.monotonic() - started,\n        "history": {key: parent["history"][key] + history[key] for key in history},\n        "stage_boundary": parent["stage_boundary"],\n        "extension_boundary": len(parent["history"]["accuracy"]),\n        "parent_last_sha256": file_hash(source),\n        "optimizer_restored": True,\n        "extension_best_val": max(history["val_accuracy"]),\n        "stage1_best_val": parent["stage1_best_val"],\n        "stage2_best_val": parent["stage2_best_val"],\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef select_winner(records):\n    # Stable order preserves earlier experiments on exact ties.\n    return max(records, key=lambda record: record["validation"]["accuracy"])\n', 'common_workflow': '"""Standalone Colab orchestration; shared contracts, audit gates and artifacts."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport platform\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import common_engine as engine\n\nEXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}\n\n\ndef signature(value):\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, ensure_ascii=False).encode()\n    ).hexdigest()\n\n\ndef extract_zip(source, destination):\n    """Validate every member before creating any output; no overwrite."""\n    destination = Path(destination).resolve()\n    if destination.exists():\n        raise FileExistsError(destination)\n    with zipfile.ZipFile(source) as archive:\n        seen = set()\n        for item in archive.infolist():\n            name = item.orig_filename\n            path = PurePosixPath(name)\n            if (\n                path.is_absolute()\n                or ".." in path.parts\n                or "\\\\" in name\n                or ":" in name\n                or stat.S_ISLNK(item.external_attr >> 16)\n            ):\n                raise ValueError("Unsafe ZIP member: " + name)\n            key = name.rstrip("/").casefold()\n            if key in seen:\n                raise ValueError("Duplicate ZIP destination: " + name)\n            seen.add(key)\n        destination.mkdir(parents=True)\n        archive.extractall(destination)\n\n\ndef roots_for(extracted):\n    roots = {}\n    for kind in ("original", "augmented"):\n        matches = [\n            p\n            for p in Path(extracted).rglob("*")\n            if p.is_dir()\n            and p.name.lower() == kind\n            and all((p / s).is_dir() for s in ("train", "val", "test"))\n        ]\n        if len(matches) != 1:\n            raise ValueError(f"{kind}/train,val,test 구조를 하나로 확인하세요: {matches}")\n        roots[kind] = matches[0]\n    return roots\n\n\ndef verify_audit(directory, domain, classes, digest):\n    directory = Path(directory)\n    manifest = engine.read_json(directory / "audit_manifest.json")\n    for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv"):\n        if engine.file_hash(directory / name) != manifest[name]:\n            raise ValueError("검증 보고서가 변경됐습니다: " + name)\n    audit = engine.read_json(directory / "audit_summary.json")\n    if (\n        audit["domain"] != domain\n        or audit["class_names"] != classes\n        or audit["data_sha256"] != digest\n        or audit.get("protocol") != "common_audit_v1"\n        or audit["status"] != "mechanical_checks_passed_with_limitations"\n    ):\n        raise ValueError("대상/클래스/데이터가 다르거나 검증 문제가 있습니다. 공통 ①을 확인하세요.")\n    return audit\n\n\ndef prepare(config, profiles, sources, commit, local_parent="/content"):\n    c = dict(config)\n    c.setdefault("expected_data_sha256", "")\n    c.setdefault("seeds", [c["seed"]])\n    if c["domain"] not in profiles or c["mode"] not in (\n        "audit",\n        "comparison",\n        "suite",\n        "baseline3",\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "final_candidate",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n        "web_skin_pmg_final",\n    ):\n        raise ValueError("DOMAIN/MODE 설정을 확인하세요.")\n    if c["train_variant"] not in ("original", "augmented"):\n        raise ValueError("TRAIN_VARIANT는 original 또는 augmented입니다.")\n    for key in ("batch_size", "epochs1", "epochs2", "extension_epochs"):\n        if not isinstance(c[key], int) or c[key] <= 0:\n            raise ValueError(key + "는 양의 정수여야 합니다.")\n    if (\n        not isinstance(c["seeds"], list)\n        or not c["seeds"]\n        or any(not isinstance(value, int) or value < 0 for value in c["seeds"])\n        or len(set(c["seeds"])) != len(c["seeds"])\n    ):\n        raise ValueError("SEEDS는 서로 다른 0 이상의 정수 목록이어야 합니다.")\n    if c["mode"] == "baseline3" and len(c["seeds"]) != 3:\n        raise ValueError("baseline3는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "paper_suite" and len(c["seeds"]) != 3:\n        raise ValueError("paper_suite는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "supcon_compare" and c["seeds"] != [42]:\n        raise ValueError("supcon_compare의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] == "supcon_repeat" and c["seeds"] != [43, 44]:\n        raise ValueError("supcon_repeat의 확인 seed는 [43, 44]여야 합니다.")\n    if c["mode"] == "paper_screen" and c["seeds"] != [42]:\n        raise ValueError("paper_screen의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] in (\n        "sam_screen",\n        "final_candidate",\n        "web_skin_pmg_final",\n    ) and c["seeds"] != [42]:\n        raise ValueError(c["mode"] + "의 seed는 [42]여야 합니다.")\n    project = Path(c["project_root"])\n    if not project.is_dir():\n        raise FileNotFoundError("PROJECT_ROOT 폴더를 확인하세요: " + str(project))\n    if c["mode"] != "audit" and not (\n        c["audit_dir"] or c["expected_data_sha256"]\n    ):\n        raise ValueError("AUDIT_DIR 또는 확인된 EXPECTED_DATA_SHA256을 입력하세요.")\n    if c["data_zip"]:\n        candidates = [Path(c["data_zip"])]\n    else:\n        candidates = sorted(\n            p\n            for p in (project / "datasets").rglob(c["domain"] + "*")\n            if p.is_file() and zipfile.is_zipfile(p)\n        )\n    if len(candidates) != 1 or not zipfile.is_zipfile(candidates[0]):\n        raise ValueError(f"DATA_ZIP으로 ZIP 하나를 지정하세요: {candidates}")\n    source = candidates[0]\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]\n    local = Path(local_parent) / ("mediflow_" + run_id)\n    local.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(source) as archive:\n        required = (\n            source.stat().st_size + sum(m.file_size for m in archive.infolist()) + 2 * 1024**3\n        )\n    if required > shutil.disk_usage(local).free:\n        raise RuntimeError("Colab 압축 해제 공간이 부족합니다.")\n    copied = local / "input.zip"\n    shutil.copyfile(source, copied)\n    digest = engine.file_hash(copied)\n    if digest != engine.file_hash(source):\n        raise OSError("Drive ZIP 복사 내용 불일치")\n    classes = profiles[c["domain"]]\n    audit = None\n    if c["mode"] != "audit":\n        if c["audit_dir"]:\n            audit = verify_audit(c["audit_dir"], c["domain"], classes, digest)\n        else:\n            expected = c["expected_data_sha256"].strip().lower()\n            if len(expected) != 64 or any(ch not in "0123456789abcdef" for ch in expected):\n                raise ValueError("EXPECTED_DATA_SHA256은 64자리 SHA-256이어야 합니다.")\n            if digest != expected:\n                raise ValueError("DATA_ZIP이 확인된 SHA-256과 다릅니다.")\n            audit = {\n                "domain": c["domain"],\n                "class_names": classes,\n                "data_sha256": digest,\n                "protocol": "expected_sha256_v1",\n                "status": "independent_audit_skipped",\n                "limitations": [\n                    "Independent common audit was skipped by the project owner",\n                    "Person, lesion and capture-session leakage remains unverified",\n                    "Perceptual near-duplicate and clinical label checks were not performed",\n                ],\n            }\n    extract_zip(copied, local / "dataset")\n    settings = {\n        k: c[k]\n        for k in (\n            "domain",\n            "mode",\n            "seed",\n            "seeds",\n            "batch_size",\n            "epochs1",\n            "epochs2",\n            "extension_epochs",\n            "train_variant",\n        )\n    }\n    settings.update(\n        classes=classes,\n        data_sha256=digest,\n        source_hashes={k: signature(v) for k, v in sources.items()},\n        protocol="common_v1",\n        audit=signature(audit),\n        environment=dict(\n            tensorflow=tf.__version__,\n            keras=keras.__version__,\n            numpy=np.__version__,\n            python=platform.python_version(),\n        ),\n    )\n    if c["mode"] in (\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n    ):\n        settings["experiments"] = c.get("experiments", [])\n    if c["mode"] in ("sam_screen", "final_candidate", "web_skin_pmg_final"):\n        settings.update(\n            parent_run_dir=c.get("parent_run_dir"),\n            parent_model_sha256=c.get(\n                "parent_model_sha256", c.get("parent_stage1_sha256")\n            ),\n        )\n    if c["mode"] == "sam_screen":\n        settings["sam_rho"] = c.get("sam_rho")\n    if c["mode"] in ("web_skin_wsdan", "web_skin_paper_suite"):\n        settings.update(\n            attention_maps=c.get("attention_maps"),\n            crop_threshold=c.get("crop_threshold"),\n            drop_threshold=c.get("drop_threshold"),\n        )\n    if c["mode"] in ("web_skin_paper_suite", "web_skin_pmg_b1_384"):\n        settings.update(\n            pmg_jigsaw_grids=c.get("pmg_jigsaw_grids"),\n        )\n    if c["mode"] == "web_skin_paper_suite":\n        settings.update(\n            mixstyle_alpha=c.get("mixstyle_alpha"),\n            mixstyle_probability=c.get("mixstyle_probability"),\n        )\n    sig = signature(settings)\n    if c["resume_dir"]:\n        output = Path(c["resume_dir"])\n        if c["mode"] == "audit":\n            raise ValueError("검사는 새 실행으로 시작하세요. RESUME_DIR을 비우세요.")\n        if engine.read_json(output / "run_config.json")["signature"] != sig:\n            raise ValueError(\n                "코드/설정/환경/데이터/검증이 다른 실행입니다. 새 결과 폴더를 사용하세요."\n            )\n    else:\n        output = project / "2_results" / c["domain"] / (c["mode"] + "_" + run_id)\n        output.mkdir(parents=True, exist_ok=False)\n        engine.write_json(\n            output / "run_config.json",\n            {\n                "signature": sig,\n                "settings": settings,\n                "code_commit_at_generation": commit,\n                "code_state": "embedded sources include uncommitted changes; exact sources saved",\n                "source_zip": str(source),\n                "audit_source": c["audit_dir"] or "expected_data_sha256_only",\n                "baseline": (\n                    (\n                        "Fixed PMG B0/256 Validation candidate reused; model not retrained"\n                        if c["mode"] == "web_skin_pmg_final"\n                        else "Saved B0/256/CE Validation metrics reused; baseline not retrained"\n                    )\n                    if c["mode"]\n                    in (\n                        "web_skin_wsdan",\n                        "web_skin_paper_suite",\n                        "web_skin_pmg_b1_384",\n                        "web_skin_pmg_final",\n                    )\n                    else "ImageNet pretrained EfficientNet; historical metrics not reused"\n                ),\n                "gpu": [str(d) for d in tf.config.list_physical_devices("GPU")],\n            },\n        )\n        for name, code in sources.items():\n            (output / (name + ".py")).write_text(code, encoding="utf-8")\n        engine.write_json(output / "class_names.json", classes)\n        if c["audit_dir"]:\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "audit_summary.json", output / "audit_summary.json"\n            )\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "image_inventory.csv", output / "image_inventory.csv"\n            )\n        elif audit:\n            engine.write_json(output / "audit_summary.json", audit)\n    context = dict(\n        config=c,\n        classes=classes,\n        signature=sig,\n        output=output,\n        local=local,\n        data_hash=digest,\n        audit=audit,\n        extracted=local / "dataset",\n        project=project,\n    )\n    if c["mode"] != "audit":\n        context["roots"] = roots_for(context["extracted"])\n        # Dataset ZIP is immutable and matches the audit; verify class folders again.\n        for root in context["roots"].values():\n            for split in ("train", "val", "test"):\n                found = sorted(p.name for p in (root / split).iterdir() if p.is_dir())\n                if found != sorted(classes):\n                    raise ValueError(f"클래스 불일치: {root / split}")\n    print("실행 결과:", output)\n    return context\n\n\ndef archive_results(context):\n    output = context["output"]\n    destination = output.parent / (output.name + "_results_" + uuid.uuid4().hex[:8] + ".zip")\n    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in sorted(output.rglob("*")):\n            if p.is_file() and p.suffix not in (".keras", ".tmp"):\n                archive.write(p, output.name + "/" + p.relative_to(output).as_posix())\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("보고서 ZIP 검사 실패")\n    print("로컬로 내려받을 결과 ZIP:", destination)\n    return destination\n\n\ndef audit_run(context):\n    from mediflow_datasets.common_audit import audit_dataset\n\n    summary = audit_dataset(\n        context["extracted"],\n        context["output"],\n        context["config"]["domain"],\n        context["classes"],\n        context["data_hash"],\n    )\n    engine.write_json(\n        context["output"] / "audit_manifest.json",\n        {\n            name: engine.file_hash(context["output"] / name)\n            for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv")\n        },\n    )\n    archive_results(context)\n    print("검사 상태:", summary["status"], "\\n학습 AUDIT_DIR:", context["output"])\n    return summary\n\n\ndef factory(context, variant, seed=None):\n    shuffle_seed = context["config"]["seed"] if seed is None else seed\n\n    def load(split, size, shuffle):\n        # All trials use exactly the same ORIGINAL validation and test images.\n        root = context["roots"][variant if split == "train" else "original"]\n        ds = keras.utils.image_dataset_from_directory(\n            root / split,\n            class_names=context["classes"],\n            label_mode="categorical",\n            image_size=(size, size),\n            interpolation="bilinear",\n            batch_size=context["config"]["batch_size"],\n            shuffle=shuffle,\n            seed=shuffle_seed if shuffle else None,\n        )\n        paths = [Path(p).relative_to(context["extracted"]).as_posix() for p in ds.file_paths]\n        return ds.prefetch(tf.data.AUTOTUNE), paths\n\n    return load\n\n\ndef run(context):\n    c, output = context["config"], context["output"]\n    if c["mode"] == "comparison":\n        trials = [\n            dict(id=kind, backbone="B0", size=224, loss="ce", variant=kind)\n            for kind in ("original", "augmented")\n        ]\n    else:\n        trials = [dict(spec, variant=c["train_variant"]) for spec in engine.TRIALS]\n    records = []\n    try:\n        for spec in trials:\n            spec["class_count"] = len(context["classes"])\n            spec["class_names"] = context["classes"]\n            load = factory(context, spec["variant"])\n            record = engine.run_trial(\n                spec,\n                load,\n                output,\n                context["signature"],\n                c["seed"],\n                c["epochs1"],\n                0 if c["mode"] == "comparison" else c["epochs2"],\n            )\n            records.append(record)\n            engine.write_json(output / "progress.json", {"completed": [r["id"] for r in records]})\n        if c["mode"] == "suite":\n            parent = records[-1]\n            records.append(\n                engine.extend_b1(\n                    parent,\n                    factory(context, c["train_variant"]),\n                    output,\n                    context["signature"],\n                    c["seed"],\n                    c["extension_epochs"],\n                )\n            )\n        engine.write_json(output / "all_validation_results.json", records)\n        return records\n    except Exception as exc:\n        engine.write_json(\n            output / ("failure_" + uuid.uuid4().hex[:8] + ".json"),\n            {\n                "error": repr(exc),\n                "completed": [r["id"] for r in records],\n                "resume_dir": str(output),\n            },\n        )\n        print("중단. 완료된 실험을 유지합니다. RESUME_DIR:", output)\n        raise\n\n\ndef confusion(ax, metrics, title):\n    cm = np.asarray(metrics["confusion_matrix"])\n    ax.imshow(cm, cmap="Blues")\n    codes = [f"C{i}" for i in range(len(cm))]\n    ax.set(\n        title=title,\n        xlabel="Predicted",\n        ylabel="True",\n        xticks=range(len(cm)),\n        yticks=range(len(cm)),\n        xticklabels=codes,\n        yticklabels=codes,\n    )\n    for i in range(len(cm)):\n        for j in range(len(cm)):\n            ax.text(\n                j,\n                i,\n                str(cm[i, j]),\n                ha="center",\n                va="center",\n                fontsize=8,\n                color="white" if cm[i, j] > cm.max() / 2 else "black",\n            )\n\n\ndef errors(context, csv_path, destination):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n    from PIL import Image\n\n    frame = pd.read_csv(csv_path)\n    wrong = frame[frame.true_index != frame.pred_index].head(8)\n    fig, axes = plt.subplots(2, 4, figsize=(14, 7))\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, (_, row) in zip(axes.flat, wrong.iterrows(), strict=False):\n        path = (context["extracted"] / row["path"]).resolve()\n        if not path.is_relative_to(context["extracted"].resolve()):\n            raise ValueError("Prediction path escapes dataset")\n        with Image.open(path) as image:\n            ax.imshow(image.convert("RGB"))\n        ax.set_title(f"True C{row.true_index} / Pred C{row.pred_index}")\n    if wrong.empty:\n        fig.suptitle("No misclassifications")\n    fig.tight_layout()\n    fig.savefig(destination, dpi=160)\n    plt.close(fig)\n\n\ndef overview(context, records):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n\n    output = context["output"]\n    fig, axes = plt.subplots(\n        (len(records) + 1) // 2, 4, figsize=(24, 4.5 * ((len(records) + 1) // 2)), squeeze=False\n    )\n    for index, record in enumerate(records):\n        row, col = divmod(index, 2)\n        for offset, metric in enumerate(("accuracy", "loss")):\n            ax = axes[row, col * 2 + offset]\n            h = record["history"]\n            x = np.arange(1, len(h[metric]) + 1)\n            ax.plot(x, h[metric], label="Train")\n            ax.plot(x, h["val_" + metric], label="Validation")\n            if record["stage_boundary"] < len(x):\n                ax.axvline(record["stage_boundary"] + 0.5, ls="--", color="gray")\n            if "extension_boundary" in record:\n                ax.axvline(record["extension_boundary"] + 0.5, ls=":", color="green")\n            ax.set(title=record["id"] + " / " + metric, xlabel="Epoch", ylabel=metric)\n            if metric == "accuracy":\n                ax.set_ylim(0, 1)\n            ax.grid(alpha=0.25)\n            ax.legend()\n        directory = output / record["id"] / record["attempt"]\n        errors(\n            context, directory / "validation_predictions.csv", directory / "validation_errors.png"\n        )\n    fig.suptitle(context["config"]["domain"] + " / Loss definitions differ across CE, LS, Focal")\n    fig.tight_layout()\n    fig.savefig(output / "all_training_curves.png", dpi=180)\n    fig.savefig(output / "all_training_curves.pdf")\n    plt.show()\n    plt.close(fig)\n    table = pd.DataFrame(\n        [\n            dict(\n                experiment=r["id"],\n                selected_stage=r["selected_stage"],\n                validation_accuracy=r["validation"]["accuracy"],\n                validation_macro_f1=r["validation"]["macro_f1"],\n                parameters=r["parameters"],\n                seconds_this_trial=r["training_seconds"],\n                epochs=len(r["history"]["accuracy"]),\n                model_bytes=r["model_bytes"],\n            )\n            for r in records\n        ]\n    )\n    table.to_csv(output / "experiment_comparison.csv", index=False, encoding="utf-8-sig")\n    print(table.to_string(index=False))\n    fig, axes = plt.subplots(2, 1, figsize=(14, 11))\n    x = np.arange(len(records))\n    axes[0].bar(x - 0.2, table.validation_accuracy, 0.4, label="Validation Accuracy")\n    axes[0].bar(x + 0.2, table.validation_macro_f1, 0.4, label="Validation Macro F1")\n    axes[0].set(xticks=x, xticklabels=table.experiment, ylim=(0, 1))\n    axes[0].tick_params(axis="x", labelrotation=15)\n    axes[0].legend()\n    matrix = np.array([r["validation"]["class_f1"] for r in records])\n    axes[1].imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")\n    axes[1].set(\n        xticks=range(len(context["classes"])),\n        xticklabels=[f"C{i}" for i in range(len(context["classes"]))],\n        yticks=x,\n        yticklabels=table.experiment,\n        title="Validation class F1",\n    )\n    for i in range(len(records)):\n        for j in range(len(context["classes"])):\n            axes[1].text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=7)\n    fig.tight_layout()\n    fig.savefig(output / "validation_performance_dashboard.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    fig, axes = plt.subplots(\n        (len(records) + 2) // 3, 3, figsize=(18, 6 * ((len(records) + 2) // 3)), squeeze=False\n    )\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, r in zip(axes.flat, records, strict=False):\n        ax.axis("on")\n        confusion(ax, r["validation"], r["id"])\n    fig.tight_layout()\n    fig.savefig(output / "all_validation_confusion_matrices.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n\ndef finish(context, records):\n    import matplotlib.pyplot as plt\n\n    output = context["output"]\n    overview(context, records)\n    winner = engine.select_winner(records)\n    model_path = engine.selected_model_path(output, winner)\n    selection = dict(\n        winner=winner["id"],\n        model_sha256=engine.file_hash(model_path),\n        signature=context["signature"],\n        validation=winner["validation"],\n    )\n    selected_file = output / "selection_before_test.json"\n    if selected_file.exists() and engine.read_json(selected_file) != selection:\n        raise ValueError("이미 고정한 선택 모델이 다릅니다.")\n    if not selected_file.exists():\n        engine.write_json(selected_file, selection)\n    marker = output / "test_completed.json"\n    if marker.exists():\n        tested = engine.read_json(marker)\n        if tested["selection"] != selection:\n            raise ValueError("기존 Test 모델과 다릅니다.")\n        for name, digest in tested["hashes"].items():\n            if engine.file_hash(output / name) != digest:\n                raise ValueError("Test 파일이 변경됐습니다.")\n        metrics = tested["metrics"]\n    else:\n        keras.backend.clear_session()\n        model = keras.models.load_model(model_path, compile=False)\n        ds, paths = factory(context, winner["spec"]["variant"])(\n            "test", winner["spec"]["size"], False\n        )\n        metrics = engine.evaluate_to_files(model, ds, paths, output, "final_test")\n        engine.write_json(\n            marker,\n            dict(\n                selection=selection,\n                metrics=metrics,\n                hashes={\n                    name: engine.file_hash(output / name)\n                    for name in ("final_test_metrics.json", "final_test_predictions.csv")\n                },\n            ),\n        )\n        del model\n    fig, ax = plt.subplots(figsize=(8, 8))\n    confusion(ax, metrics, winner["id"] + " / Final Test")\n    fig.tight_layout()\n    fig.savefig(output / "final_test_confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    errors(context, output / "final_test_predictions.csv", output / "final_test_errors.png")\n    card = dict(\n        domain=context["config"]["domain"],\n        class_names=context["classes"],\n        normal_included="정상" in context["classes"],\n        validation=winner["validation"],\n        test=metrics,\n        selected_model=winner["id"],\n        model_sha256=selection["model_sha256"],\n        input_size=winner["spec"]["size"],\n        data_sha256=context["data_hash"],\n        status="public_data_candidate_not_device_validated",\n        limitations=context["audit"]["limitations"]\n        + [\n            "Single seed; small differences are not established as robust gains",\n            "Out-of-scope rejection absent; scores are not calibrated correctness",\n        ],\n    )\n    engine.write_json(output / "model_card.json", card)\n    package_parent = (\n        context["project"] / "2_results" / context["config"]["domain"] / "selected_models"\n    )\n    package = package_parent / (output.name + "_" + uuid.uuid4().hex[:8])\n    package.mkdir(parents=True, exist_ok=False)\n    shutil.copyfile(model_path, package / "model.keras")\n    if engine.file_hash(package / "model.keras") != selection["model_sha256"]:\n        raise OSError("모델 복사 불일치")\n    for name in (\n        "class_names.json",\n        "model_card.json",\n        "run_config.json",\n        "selection_before_test.json",\n        "audit_summary.json",\n        "final_test_metrics.json",\n        "common_engine.py",\n        "common_audit.py",\n        "common_workflow.py",\n    ):\n        shutil.copyfile(output / name, package / name)\n    engine.write_json(\n        package / "preprocessing.json",\n        dict(\n            input_shape=[winner["spec"]["size"], winner["spec"]["size"], 3],\n            color="RGB",\n            dtype="float32",\n            pixel_range=[0, 255],\n            external_normalization=False,\n            internal_rescaling="1/255",\n            resize="TensorFlow bilinear; no crop/pad; antialias=False",\n            exif_transpose=False,\n            output="softmax scores in class_names.json order",\n        ),\n    )\n    engine.write_json(\n        package / "manifest.json",\n        {p.name: engine.file_hash(p) for p in package.iterdir() if p.is_file()},\n    )\n    destination = Path(shutil.make_archive(str(package), "zip", package.parent, package.name))\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("모델 ZIP 손상")\n        manifest = engine.read_json(package / "manifest.json")\n        for name, digest in manifest.items():\n            if hashlib.sha256(archive.read(package.name + "/" + name)).hexdigest() != digest:\n                raise OSError("ZIP 내용 불일치: " + name)\n    destination.with_suffix(".zip.sha256").write_text(\n        engine.file_hash(destination), encoding="ascii"\n    )\n    archive_results(context)\n    print("선정 모델:", winner["id"], "\\nTest:", metrics, "\\n후보 ZIP:", destination)\n    return card\n', 'web_skin_pmg_final': '"""One-time Test evaluation and packaging of the fixed Web Skin PMG B0/256 model."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport shutil\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\n\nfrom mediflow_datasets import common_engine as engine\nfrom mediflow_datasets import common_workflow as flow\n\nPROTOCOL = "web_skin_pmg_b0_256_final_candidate_v1"\nPARENT_TRIAL = "pmg_b0_256_ce_seed_42"\nMODEL_NAME = "stage2_best.keras"\nEXPECTED_MODEL_SHA256 = "83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4"\nEXPECTED_DATA_SHA256 = "f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d"\nEXPECTED_CLASSES = ["건선", "아토피", "여드름", "정상", "주사"]\nEXPECTED_VALIDATION = {\n    "accuracy": 0.85,\n    "macro_f1": 0.8473279632397033,\n    "count": 500,\n}\n\n\ndef validate(context):\n    config = context["config"]\n    if config["domain"] != "web_skin" or config["mode"] != "web_skin_pmg_final":\n        raise ValueError("이 노트북은 Web Skin PMG·B0·256 최종평가 전용입니다.")\n    if config["train_variant"] != "augmented":\n        raise ValueError("고정된 PMG 후보와 같은 augmented 데이터 계약이 필요합니다.")\n    if config["seed"] != 42 or config["seeds"] != [42]:\n        raise ValueError("고정된 PMG 후보의 seed 계약이 다릅니다.")\n    if context["classes"] != EXPECTED_CLASSES:\n        raise ValueError("Web Skin 클래스 순서가 고정 후보와 다릅니다.")\n    if context["data_hash"] != EXPECTED_DATA_SHA256:\n        raise ValueError("Web Skin 데이터 SHA-256이 고정 후보와 다릅니다.")\n    parent = config.get("parent_run_dir")\n    if not isinstance(parent, str) or not parent.strip():\n        raise ValueError("PARENT_RUN_DIR을 입력하세요.")\n    digest = config.get("parent_model_sha256")\n    if digest != EXPECTED_MODEL_SHA256:\n        raise ValueError("고정된 PMG·B0·256 모델 SHA-256과 다릅니다.")\n\n\ndef _parent(context):\n    parent = Path(context["config"]["parent_run_dir"])\n    marker = parent / PARENT_TRIAL / "completed.json"\n    if not marker.is_file():\n        raise FileNotFoundError("PMG·B0·256 완료 기록을 찾을 수 없습니다: " + str(marker))\n    record = engine.read_json(marker)\n    spec = record.get("spec", {})\n    validation = record.get("validation", {})\n    if (\n        record.get("id") != PARENT_TRIAL\n        or record.get("selected_stage") != "stage2"\n        or record.get("selected_model") != MODEL_NAME\n        or record.get("test_evaluated", False) is not False\n        or spec.get("method") != "efficientnet_b0_pmg_adaptation"\n        or spec.get("input_size") != 256\n        or spec.get("jigsaw_grids") != [8, 4, 2]\n        or validation.get("accuracy") != EXPECTED_VALIDATION["accuracy"]\n        or validation.get("macro_f1") != EXPECTED_VALIDATION["macro_f1"]\n        or validation.get("count") != EXPECTED_VALIDATION["count"]\n    ):\n        raise ValueError("고정한 PMG·B0·256 Validation 후보 기록과 다릅니다.")\n    model_path = marker.parent / record["attempt"] / MODEL_NAME\n    actual = engine.file_hash(model_path)\n    relative = record["attempt"] + "/" + MODEL_NAME\n    if (\n        actual != EXPECTED_MODEL_SHA256\n        or record.get("artifact_hashes", {}).get(relative) != actual\n    ):\n        raise ValueError("PMG·B0·256 최종 후보 모델 SHA-256이 다릅니다.")\n    settings = engine.read_json(parent / "run_config.json")["settings"]\n    if (\n        settings["data_sha256"] != context["data_hash"]\n        or settings["classes"] != context["classes"]\n        or settings["seed"] != 42\n    ):\n        raise ValueError("부모 실행의 데이터·클래스·seed가 현재 실행과 다릅니다.")\n    return record, model_path, actual\n\n\ndef _predictor(model):\n    if not isinstance(model.outputs, list) or len(model.outputs) != 4:\n        raise ValueError("PMG는 네 개의 logit 출력이 필요합니다.")\n    total = keras.layers.Add(name="pmg_logit_sum")(model.outputs)\n    probabilities = keras.layers.Activation("softmax", name="predictions")(total)\n    return keras.Model(model.input, probabilities, name="web_skin_pmg_inference")\n\n\ndef _model_contract(model):\n    shapes = [tuple(shape) for shape in model.output_shape]\n    if (\n        tuple(model.input_shape) != (None, 256, 256, 3)\n        or shapes != [(None, 5)] * 4\n        or model.count_params() != 8_872_375\n    ):\n        raise ValueError("PMG·B0·256 모델 입출력 또는 파라미터 계약이 다릅니다.")\n    try:\n        backbone = model.get_layer("pmg_backbone")\n    except ValueError as exc:\n        raise ValueError("PMG Backbone을 찾지 못했습니다.") from exc\n    rescaling = [\n        layer for layer in backbone.layers if isinstance(layer, keras.layers.Rescaling)\n    ]\n    if not any(\n        np.asarray(layer.scale).size == 1\n        and np.isclose(float(np.asarray(layer.scale).item()), 1 / 255)\n        and np.allclose(layer.offset, 0)\n        for layer in rescaling\n    ):\n        raise ValueError("모델 내부 Rescaling(1/255)을 찾지 못했습니다.")\n    values = np.broadcast_to(\n        np.array([0, 127.5, 255], dtype="float32")[:, None, None, None],\n        (3, 256, 256, 3),\n    ).copy()\n    raw = model(values, training=False)\n    if len(raw) != 4 or any(\n        np.asarray(output).shape != (3, 5) or not np.isfinite(output).all()\n        for output in raw\n    ):\n        raise ValueError("PMG 내부 logit 출력 검사가 실패했습니다.")\n    scores = np.asarray(_predictor(model)(values, training=False))\n    if (\n        scores.shape != (3, 5)\n        or not np.isfinite(scores).all()\n        or np.any(scores < 0)\n        or np.any(scores > 1)\n        or not np.allclose(scores.sum(axis=1), 1, atol=1e-5)\n    ):\n        raise ValueError("PMG 합산 softmax 출력 검사가 실패했습니다.")\n    return {\n        "input_shape": [256, 256, 3],\n        "internal_output_count": 4,\n        "public_output_count": 5,\n        "parameters": model.count_params(),\n        "dummy_forward_passed": True,\n    }\n\n\ndef _evaluate_once(context, record, model_path, model_hash):\n    output = Path(context["output"])\n    selection = {\n        "protocol": PROTOCOL,\n        "winner": PARENT_TRIAL,\n        "selection_reason": "deployment balance of validation quality and inference cost",\n        "parent_attempt": record["attempt"],\n        "parent_selected_model": MODEL_NAME,\n        "model_sha256": model_hash,\n        "validation": record["validation"],\n        "test_was_unread_at_selection": True,\n    }\n    selection_path = output / "selection_before_test.json"\n    if selection_path.exists() and engine.read_json(selection_path) != selection:\n        raise ValueError("이미 고정한 후보와 현재 후보가 다릅니다.")\n    if not selection_path.exists():\n        engine.write_json(selection_path, selection)\n    marker = output / "test_completed.json"\n    if marker.exists():\n        completed = engine.read_json(marker)\n        if completed["selection"] != selection:\n            raise ValueError("기존 Test 평가의 후보가 다릅니다.")\n        for name, digest in completed["hashes"].items():\n            if engine.file_hash(output / name) != digest:\n                raise ValueError("저장된 Test 결과가 변경됐습니다: " + name)\n        return completed["metrics"], completed["model_contract"]\n    keras.backend.clear_session()\n    model = keras.models.load_model(model_path, compile=False)\n    contract = _model_contract(model)\n    predictor = _predictor(model)\n    test, paths = flow.factory(context, "augmented", seed=42)("test", 256, False)\n    metrics = engine.evaluate_to_files(predictor, test, paths, output, "final_test")\n    del predictor, model\n    keras.backend.clear_session()\n    names = ("final_test_metrics.json", "final_test_predictions.csv")\n    engine.write_json(\n        marker,\n        {\n            "selection": selection,\n            "metrics": metrics,\n            "model_contract": contract,\n            "hashes": {name: engine.file_hash(output / name) for name in names},\n        },\n    )\n    return metrics, contract\n\n\ndef _figures(context, metrics):\n    import matplotlib.pyplot as plt\n\n    output = Path(context["output"])\n    fig, axis = plt.subplots(figsize=(8, 8))\n    flow.confusion(axis, metrics, "Web Skin PMG·B0·256 / Final Test")\n    fig.tight_layout()\n    fig.savefig(output / "final_test_confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    flow.errors(\n        context,\n        output / "final_test_predictions.csv",\n        output / "final_test_errors.png",\n    )\n\n\ndef _inference_source():\n    return \'\'\'"""MediFlow Web Skin PMG inference helper."""\nfrom pathlib import Path\n\nimport keras\nimport tensorflow as tf\n\n\ndef load_predictor(model_path):\n    model = keras.models.load_model(Path(model_path), compile=False)\n    if not isinstance(model.outputs, list) or len(model.outputs) != 4:\n        raise ValueError("Expected four PMG logit outputs")\n    total = keras.layers.Add(name="pmg_logit_sum")(model.outputs)\n    probabilities = keras.layers.Activation("softmax", name="predictions")(total)\n    return keras.Model(model.input, probabilities, name="web_skin_pmg_inference")\n\n\ndef load_image(image_path):\n    content = tf.io.read_file(str(image_path))\n    image = tf.io.decode_image(content, channels=3, expand_animations=False)\n    image = tf.image.resize(tf.cast(image, tf.float32), (256, 256), method="bilinear")\n    return image[None, ...]\n\n\ndef predict_file(predictor, image_path):\n    return predictor(load_image(image_path), training=False).numpy()[0]\n\'\'\'\n\n\ndef _package(context, record, model_path, model_hash, metrics, contract):\n    output = Path(context["output"])\n    marker_path = output / "package_completed.json"\n    if marker_path.exists():\n        marker = engine.read_json(marker_path)\n        for path_text, digest in marker["hashes"].items():\n            if engine.file_hash(Path(path_text)) != digest:\n                raise ValueError("완료된 후보 패키지가 변경됐습니다: " + path_text)\n        return Path(marker["package"]), Path(marker["archive"])\n    card = {\n        "protocol": PROTOCOL,\n        "status": "public_data_candidate_not_device_validated",\n        "domain": "web_skin",\n        "class_names": context["classes"],\n        "normal_class_included": True,\n        "architecture": "PMG with ImageNet EfficientNet-B0",\n        "input_size": [256, 256],\n        "loss": "Categorical Crossentropy",\n        "optimizer": "Adam",\n        "stage1_epochs": 15,\n        "stage2_epochs": 10,\n        "selected_model": PARENT_TRIAL,\n        "selection_reason": "validation quality and deployment cost balance",\n        "model_sha256": model_hash,\n        "model_parameter_count": contract["parameters"],\n        "validation": record["validation"],\n        "test": metrics,\n        "data_zip_sha256": context["data_hash"],\n        "limitations": context["audit"]["limitations"]\n        + [\n            "Actual webcam patient images were not evaluated",\n            "No out-of-scope rejection mechanism",\n            "Softmax scores are not calibrated correctness probabilities",\n        ],\n    }\n    engine.write_json(output / "model_card.json", card)\n    package_parent = context["project"] / "2_results" / "web_skin" / "selected_models"\n    suffix = output.name.removeprefix("web_skin_pmg_final_")\n    name = "public_candidate_v2_pmg_b0_256_ce_" + suffix\n    package = package_parent / name\n    package.mkdir(parents=True, exist_ok=False)\n    shutil.copyfile(model_path, package / "web_skin_pmg_model.keras")\n    if engine.file_hash(package / "web_skin_pmg_model.keras") != model_hash:\n        raise OSError("후보 모델 복사 해시가 다릅니다.")\n    for filename in (\n        "class_names.json",\n        "audit_summary.json",\n        "run_config.json",\n        "selection_before_test.json",\n        "test_completed.json",\n        "final_test_metrics.json",\n        "final_test_predictions.csv",\n        "final_test_confusion_matrix.png",\n        "final_test_errors.png",\n        "model_card.json",\n        "common_engine.py",\n        "common_workflow.py",\n        "web_skin_pmg_final.py",\n    ):\n        shutil.copyfile(output / filename, package / filename)\n    (package / "inference.py").write_text(_inference_source(), encoding="utf-8")\n    preprocessing = {\n        "input_shape": [256, 256, 3],\n        "color_order": "RGB",\n        "input_dtype": "float32",\n        "input_pixel_range": [0, 255],\n        "external_normalization": False,\n        "internal_rescaling": "1/255",\n        "resize": "TensorFlow bilinear, antialias=False",\n        "aspect_ratio": "resize to 256x256, no crop/pad",\n        "exif_transpose": False,\n        "internal_outputs": "four PMG logits",\n        "public_output": "sum logits then 5-way softmax in class_names.json order",\n    }\n    engine.write_json(package / "preprocessing.json", preprocessing)\n    engine.write_json(package / "model_contract_check.json", contract)\n    (package / "MODEL_CARD.md").write_text(\n        "# MediFlow Web Skin 공개 데이터 후보 v2\\n\\n"\n        "PMG / EfficientNet-B0 / 256 / Cross Entropy / Adam.\\n\\n"\n        f"클래스 순서: {context[\'classes\']}\\n\\n"\n        f"Validation Accuracy: {record[\'validation\'][\'accuracy\']}\\n\\n"\n        f"Test Accuracy: {metrics[\'accuracy\']}\\n\\n"\n        f"Test Macro F1: {metrics[\'macro_f1\']}\\n\\n"\n        "입력은 얼굴 피부 RGB float32 0~255를 256×256으로 resize합니다. 외부 /255는 "\n        "금지합니다. 모델의 네 logit 출력을 직접 사용하지 말고 inference.py처럼 합산 후 "\n        "softmax를 적용해야 합니다.\\n\\n"\n        "실제 웹캠 환자 검증과 범위 밖 입력 거부 기능은 포함되지 않습니다.\\n",\n        encoding="utf-8",\n    )\n    manifest = {\n        "protocol": PROTOCOL,\n        "created_at": datetime.now(timezone.utc).isoformat(),\n        "model_file": "web_skin_pmg_model.keras",\n        "inference_file": "inference.py",\n        "model_sha256": model_hash,\n        "source_run": Path(context["config"]["parent_run_dir"]).name,\n        "source_trial": PARENT_TRIAL,\n        "validation": record["validation"],\n        "test": metrics,\n        "data_zip_sha256": context["data_hash"],\n        "test_evaluated_in_this_run": True,\n        "artifact_hashes": {\n            path.relative_to(package).as_posix(): engine.file_hash(path)\n            for path in sorted(package.rglob("*"))\n            if path.is_file()\n        },\n    }\n    engine.write_json(package / "package_manifest.json", manifest)\n    archive_path = package.with_suffix(".zip")\n    with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(package.rglob("*")):\n            if path.is_file():\n                archive.write(path, name + "/" + path.relative_to(package).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        if archive.testzip() is not None:\n            raise OSError("후보 ZIP CRC 검사가 실패했습니다.")\n        expected = {\n            **manifest["artifact_hashes"],\n            "package_manifest.json": engine.file_hash(package / "package_manifest.json"),\n        }\n        for relative, digest in expected.items():\n            actual = hashlib.sha256(archive.read(name + "/" + relative)).hexdigest()\n            if actual != digest:\n                raise OSError("후보 ZIP 내용 해시가 다릅니다: " + relative)\n    archive_hash = engine.file_hash(archive_path)\n    archive_path.with_suffix(".zip.sha256").write_text(archive_hash, encoding="ascii")\n    engine.write_json(\n        marker_path,\n        {\n            "package": str(package),\n            "archive": str(archive_path),\n            "hashes": {\n                str(package / "package_manifest.json"): engine.file_hash(\n                    package / "package_manifest.json"\n                ),\n                str(archive_path): archive_hash,\n            },\n        },\n    )\n    return package, archive_path\n\n\ndef finalize(context):\n    validate(context)\n    record, model_path, model_hash = _parent(context)\n    metrics, contract = _evaluate_once(context, record, model_path, model_hash)\n    _figures(context, metrics)\n    package, archive = _package(\n        context, record, model_path, model_hash, metrics, contract\n    )\n    report = flow.archive_results(context)\n    print("최종 후보:", package)\n    print("배포 ZIP:", archive)\n    print("보고서 ZIP:", report)\n    return metrics, package, archive, report\n'}
BUILD_COMMIT = '268e2ad22fd23ca37d6a2987ef391944b9c7fd1b'
PROFILES = {'web_skin': ['건선', '아토피', '여드름', '정상', '주사']}
package = types.ModuleType('mediflow_datasets')
package.__path__ = []
sys.modules['mediflow_datasets'] = package
for name, source in SOURCES.items():
    module = types.ModuleType('mediflow_datasets.' + name)
    sys.modules[module.__name__] = module
    exec(compile(source, name + '.py', 'exec'), module.__dict__)
from mediflow_datasets.common_workflow import prepare
from mediflow_datasets.web_skin_pmg_final import finalize


## 4. 데이터와 선택 모델 확인

데이터, 클래스 순서, 부모 실행 기록과 `stage2_best.keras` 해시가 모두 일치해야 결과 폴더가
준비됩니다. 설정이 다르면 Test를 읽기 전에 중단합니다.


In [ ]:
config = dict(
    domain=DOMAIN, project_root=PROJECT_ROOT, data_zip=DATA_ZIP,
    audit_dir=AUDIT_DIR, expected_data_sha256=EXPECTED_DATA_SHA256,
    resume_dir=RESUME_DIR, mode=MODE, seed=SEED, seeds=SEEDS,
    batch_size=BATCH_SIZE, epochs1=STAGE1_EPOCHS, epochs2=STAGE2_EPOCHS,
    extension_epochs=EXTENSION_EPOCHS, train_variant=TRAIN_VARIANT,
    parent_run_dir=PARENT_RUN_DIR, parent_model_sha256=PARENT_MODEL_SHA256,
)
context = prepare(config, PROFILES, SOURCES, BUILD_COMMIT)
print('결과 폴더 / 중단 시 RESUME_DIR:', context['output'])
print('데이터 SHA-256:', context['data_hash'])
print('선택 모델 SHA-256:', PARENT_MODEL_SHA256)


## 5. 최종 Test와 패키징

이 셀 하나가 선택 기록 고정, Test 평가, 혼동행렬·오류 이미지 생성, 모델 계약 검사와 후보
ZIP 생성을 순서대로 수행합니다. 완료되면 `selected_models`의 모델 ZIP과 현재 결과 폴더의
보고서 ZIP 경로를 출력합니다. 같은 결과 폴더로 재개하면 저장된 Test 결과를 검증해 재사용합니다.


In [ ]:
metrics, package, model_zip, report_zip = finalize(context)
metrics
